In [0]:
from pyspark.sql.functions import (
    col, when, countDistinct, sum as spark_sum,
    avg, min as spark_min, max as spark_max,
    datediff, lit, count
)

# Load Delta tables from Databricks
fact_orders = spark.table("workspace.default.fact_orders")
dim_date = spark.table("workspace.default.dim_date")
dim_customer = spark.table("workspace.default.dim_customer")

# Join fact table with date and customer dimensions
orders_enriched = (
    fact_orders
    .join(dim_date, on="date_key", how="left")
    .join(dim_customer, on="customer_key", how="left")
)

print("Orders loaded:", orders_enriched.count())

print(
    "Customers:",
    dim_customer
    .filter(col("is_current") == True)
    .select("customer_id")
    .distinct()
    .count()
)

Orders loaded: 199989
Customers: 2000


In [0]:
# Identify customers who placed orders during the churn window
churn_window_orders = (
    orders_enriched
    .filter(
        (col("date") >= "2026-08-01") &
        (col("date") <= "2026-10-24")
    )
    .select("customer_id")
    .distinct()
    .withColumn("ordered_in_window", lit(1))
)

# Get the latest customer record
all_customers = (
    dim_customer
    .filter(col("is_current") == True)
    .select("customer_id", "name", "city", "segment")
    .dropDuplicates(["customer_id"])
)

# Assign churn labels
churn_labels = (
    all_customers
    .join(churn_window_orders, on="customer_id", how="left")
    .withColumn(
        "churned",
        when(col("ordered_in_window").isNull(), 1).otherwise(0)
    )
    .drop("ordered_in_window")
)

# Check the label distribution
churn_labels.groupBy("churned").count().show()

+-------+-----+
|churned|count|
+-------+-----+
|      0| 1998|
|      1|    2|
+-------+-----+



In [0]:
from pyspark.sql.functions import (
    col, countDistinct, sum as spark_sum,
    avg, min as spark_min, max as spark_max,
    datediff, lit, round
)

# Keep only orders before the churn cutoff
historical_orders = (
    orders_enriched
    .filter(col("date") < "2026-08-01")
)

# Calculate customer-level features
customer_features = (
    historical_orders
    .groupBy("customer_id")
    .agg(
        countDistinct("order_id").alias("total_orders"),
        spark_sum("amount").alias("total_spent"),
        avg("amount").alias("average_order_value"),
        spark_sum("quantity").alias("total_quantity"),
        spark_min("date").alias("first_order_date"),
        spark_max("date").alias("last_order_date")
    )
)

# Calculate days since the customer's last order
customer_features = customer_features.withColumn(
    "recency_days",
    datediff(lit("2026-08-01"), col("last_order_date"))
)

# Add customers who have no historical orders
customer_features = (
    all_customers
    .select("customer_id")
    .join(customer_features, on="customer_id", how="left")
    .fillna({
        "total_orders": 0,
        "total_spent": 0,
        "average_order_value": 0,
        "total_quantity": 0
    })
)

# Display the features
customer_features.show(10, truncate=False)

print("Total customer feature records:", customer_features.count())

+-----------+------------+--------------------+-------------------+--------------+----------------+---------------+------------+
|customer_id|total_orders|total_spent         |average_order_value|total_quantity|first_order_date|last_order_date|recency_days|
+-----------+------------+--------------------+-------------------+--------------+----------------+---------------+------------+
|745        |142         |3119503.920000001   |21968.33746478874  |233           |2025-01-15      |2026-07-25     |7           |
|1355       |39          |756333.9799999999   |19393.178974358972 |55            |2025-01-11      |2026-07-25     |7           |
|849        |103         |2291804.39          |22250.52805825243  |181           |2025-01-14      |2026-07-22     |10          |
|37         |76          |1293206.55          |17015.875657894736 |112           |2025-01-10      |2026-07-27     |5           |
|203        |32          |839329.48           |26229.04625        |45            |2025-01-14     

In [0]:
# Combine customer features with churn labels
ml_dataset = (
    customer_features
    .join(
        churn_labels.select("customer_id", "churned"),
        on="customer_id",
        how="inner"
    )
)

# Check the final dataset
print("Total records:", ml_dataset.count())

# Check churn distribution
ml_dataset.groupBy("churned").count().show()

# Display sample records
ml_dataset.show(10, truncate=False)

Total records: 2000
+-------+-----+
|churned|count|
+-------+-----+
|      0| 1998|
|      1|    2|
+-------+-----+

+-----------+------------+--------------------+-------------------+--------------+----------------+---------------+------------+-------+
|customer_id|total_orders|total_spent         |average_order_value|total_quantity|first_order_date|last_order_date|recency_days|churned|
+-----------+------------+--------------------+-------------------+--------------+----------------+---------------+------------+-------+
|745        |142         |3119503.920000001   |21968.33746478874  |233           |2025-01-15      |2026-07-25     |7           |0      |
|1355       |39          |756333.9799999999   |19393.178974358972 |55            |2025-01-11      |2026-07-25     |7           |0      |
|849        |103         |2291804.39          |22250.52805825243  |181           |2025-01-14      |2026-07-22     |10          |0      |
|37         |76          |1293206.55          |17015.87565789

In [0]:
from pyspark.sql.functions import sum as spark_sum, col

ml_dataset.select([
    spark_sum(
        col(c).isNull().cast("int")
    ).alias(c)
    for c in ml_dataset.columns
]).show()

+-----------+------------+-----------+-------------------+--------------+----------------+---------------+------------+-------+
|customer_id|total_orders|total_spent|average_order_value|total_quantity|first_order_date|last_order_date|recency_days|churned|
+-----------+------------+-----------+-------------------+--------------+----------------+---------------+------------+-------+
|          0|           0|          0|                  0|             0|               0|              0|           0|      0|
+-----------+------------+-----------+-------------------+--------------+----------------+---------------+------------+-------+



In [0]:
from pyspark.ml.feature import VectorAssembler

# Select numerical input features
feature_columns = [
    "total_orders",
    "total_spent",
    "average_order_value",
    "total_quantity",
    "recency_days"
]

# Convert features into a single vector column
assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

model_data = assembler.transform(ml_dataset)

# Keep only features and target
model_data = model_data.select("features", "churned")

model_data.show(5, truncate=False)

+-----------------------------------------------------+-------+
|features                                             |churned|
+-----------------------------------------------------+-------+
|[142.0,3119503.920000001,21968.33746478874,233.0,7.0]|0      |
|[39.0,756333.9799999999,19393.178974358972,55.0,7.0] |0      |
|[103.0,2291804.39,22250.52805825243,181.0,10.0]      |0      |
|[76.0,1293206.55,17015.875657894736,112.0,5.0]       |0      |
|[32.0,839329.48,26229.04625,45.0,1.0]                |0      |
+-----------------------------------------------------+-------+
only showing top 5 rows


In [0]:
from pyspark.sql.functions import col

# Separate churned and non-churned customers
churned_customers = model_data.filter(col("churned") == 1)
active_customers = model_data.filter(col("churned") == 0)

# Put one churned customer in training and one in testing
churned_train = churned_customers.limit(1)
churned_test = churned_customers.subtract(churned_train)

# Split active customers
active_train, active_test = active_customers.randomSplit(
    [0.8, 0.2],
    seed=42
)

# Combine both groups
train_data = active_train.unionByName(churned_train)
test_data = active_test.unionByName(churned_test)

# Check distributions
print("Training records:", train_data.count())
train_data.groupBy("churned").count().show()

print("Testing records:", test_data.count())
test_data.groupBy("churned").count().show()

Training records: 1641
+-------+-----+
|churned|count|
+-------+-----+
|      0| 1640|
|      1|    1|
+-------+-----+

Testing records: 359
+-------+-----+
|churned|count|
+-------+-----+
|      0|  358|
|      1|    1|
+-------+-----+



In [0]:
from pyspark.ml.classification import LogisticRegression

# Model 1
lr1 = LogisticRegression(
    featuresCol="features",
    labelCol="churned",
    maxIter=50,
    regParam=0.01
)

model1 = lr1.fit(train_data)

print("Model 1 trained: regParam = 0.01")


# Model 2
lr2 = LogisticRegression(
    featuresCol="features",
    labelCol="churned",
    maxIter=50,
    regParam=0.1
)

model2 = lr2.fit(train_data)

print("Model 2 trained: regParam = 0.1")


# Model 3
lr3 = LogisticRegression(
    featuresCol="features",
    labelCol="churned",
    maxIter=50,
    regParam=1.0
)

model3 = lr3.fit(train_data)

print("Model 3 trained: regParam = 1.0")

Model 1 trained: regParam = 0.01
Model 2 trained: regParam = 0.1
Model 3 trained: regParam = 1.0


In [0]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

# Create evaluators
accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="churned",
    predictionCol="prediction",
    metricName="accuracy"
)

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="churned",
    predictionCol="prediction",
    metricName="f1"
)

precision_evaluator = MulticlassClassificationEvaluator(
    labelCol="churned",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

recall_evaluator = MulticlassClassificationEvaluator(
    labelCol="churned",
    predictionCol="prediction",
    metricName="weightedRecall"
)

# Store models
models = {
    "Model 1 (regParam=0.01)": model1,
    "Model 2 (regParam=0.1)": model2,
    "Model 3 (regParam=1.0)": model3
}

# Evaluate each model
for name, model in models.items():
    predictions = model.transform(test_data)

    accuracy = accuracy_evaluator.evaluate(predictions)
    f1 = f1_evaluator.evaluate(predictions)
    precision = precision_evaluator.evaluate(predictions)
    recall = recall_evaluator.evaluate(predictions)

    print("\n", name)
    print("Accuracy:", accuracy)
    print("F1 Score:", f1)
    print("Weighted Precision:", precision)
    print("Weighted Recall:", recall)


 Model 1 (regParam=0.01)
Accuracy: 0.9972144846796658
F1 Score: 0.9958236694988015
Weighted Precision: 0.9944367284549314
Weighted Recall: 0.9972144846796658

 Model 2 (regParam=0.1)
Accuracy: 0.9972144846796658
F1 Score: 0.9958236694988015
Weighted Precision: 0.9944367284549314
Weighted Recall: 0.9972144846796658

 Model 3 (regParam=1.0)
Accuracy: 0.9972144846796658
F1 Score: 0.9958236694988015
Weighted Precision: 0.9944367284549314
Weighted Recall: 0.9972144846796658


In [0]:
from pyspark.sql.functions import col

# Get predictions from Model 1
predictions = model1.transform(test_data)

# Confusion matrix
confusion_matrix = (
    predictions
    .groupBy("churned", "prediction")
    .count()
    .orderBy("churned", "prediction")
)

confusion_matrix.show()

# Show the actual churned customer's prediction
predictions.filter(
    col("churned") == 1
).select(
    "churned", "prediction", "probability"
).show(truncate=False)

+-------+----------+-----+
|churned|prediction|count|
+-------+----------+-----+
|      0|       0.0|  358|
|      1|       0.0|    1|
+-------+----------+-----+

+-------+----------+-----------------------------------------+
|churned|prediction|probability                              |
+-------+----------+-----------------------------------------+
|1      |0.0       |[0.9993909937321718,6.090062678282271E-4]|
+-------+----------+-----------------------------------------+



In [0]:
import os

os.environ["MLFLOW_DFS_TMP"] = (
    "/Volumes/workspace/default/online_shop_files/mlflow_tmp"
)

print("MLflow temporary directory configured.")

MLflow temporary directory configured.


In [0]:
import mlflow
import mlflow.spark

# Select Model 1
best_model = model1
best_model_name = "Logistic Regression - Model 1"

# Get predictions
best_predictions = best_model.transform(test_data)

# Calculate metrics
accuracy = accuracy_evaluator.evaluate(best_predictions)
f1 = f1_evaluator.evaluate(best_predictions)

# Start MLflow experiment
mlflow.set_experiment("/Shared/Online-Shop-Churn-Prediction")

with mlflow.start_run(run_name="Logistic_Regression_0.01"):

    # Log parameters
    mlflow.log_param("algorithm", "Logistic Regression")
    mlflow.log_param("regParam", 0.01)
    mlflow.log_param("maxIter", 50)
    mlflow.log_param("feature_count", 5)
    mlflow.log_param("training_records", train_data.count())
    mlflow.log_param("testing_records", test_data.count())

    # Log metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("weighted_f1", f1)

    # Log model
    mlflow.spark.log_model(
        best_model,
        artifact_path="churn_model"
    )

    best_run_id = mlflow.active_run().info.run_id

    print("Best model:", best_model_name)
    print("MLflow Run ID:", best_run_id)
    print("Accuracy:", accuracy)
    print("Weighted F1:", f1)

2026/11/06 19:46:47 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-13d6730e-d08d-4529-a189-71/tmpccfbpwn8/model, flavor: spark). Fall back to return ['pyspark==4.3.0']. Set logging level to DEBUG to see the full traceback. 
2026/11/06 19:46:47 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.12.0/ml/model/signatures.html for instructions on setting signature on models.


Best model: Logistic Regression - Model 1
MLflow Run ID: 34366b9a526b4d909c7b8ae2773e04f3
Accuracy: 0.9972144846796658
Weighted F1: 0.9958236694988015


In [0]:
# Save customer features as a Delta table
feature_table_name = "workspace.default.churn_features"

(
    customer_features.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(feature_table_name)
)

print("Feature table saved:", feature_table_name)

Feature table saved: workspace.default.churn_features


In [0]:
# Read the Delta table history
feature_history = spark.sql(
    "DESCRIBE HISTORY workspace.default.churn_features"
)

feature_history.select(
    "version", "timestamp", "operation"
).show()

# Capture the latest version
feature_delta_version = (
    feature_history
    .orderBy(col("version").desc())
    .first()["version"]
)

print("Feature Delta version:", feature_delta_version)

+-------+-------------------+--------------------+
|version|          timestamp|           operation|
+-------+-------------------+--------------------+
|      1|2026-11-06 19:48:33|CREATE OR REPLACE...|
|      0|2026-11-06 14:26:16|CREATE OR REPLACE...|
+-------+-------------------+--------------------+

Feature Delta version: 1


In [0]:
import mlflow

with mlflow.start_run(run_id=best_run_id):
    mlflow.log_param("feature_delta_table", feature_table_name)
    mlflow.log_param(
        "feature_delta_version",
        str(feature_delta_version)
    )

print("Feature Delta version added to MLflow.")

Feature Delta version added to MLflow.


In [0]:
import mlflow
import mlflow.spark

loaded_model = mlflow.spark.load_model(
    f"runs:/{best_run_id}/churn_model",
    dfs_tmpdir="/Volumes/workspace/default/online_shop_files/mlflow_tmp"
)

print("Model successfully reloaded!")

Model successfully reloaded!


In [0]:
# Prepare customer features for prediction
scoring_input = (
    assembler.transform(customer_features)
    .select("customer_id", "features")
)

# Generate predictions using the reloaded model
customer_predictions = loaded_model.transform(scoring_input)

# Display five customer predictions
customer_predictions.select(
    "customer_id",
    "prediction",
    "probability"
).orderBy("customer_id").show(5, truncate=False)

+-----------+----------+-----------------------------------------+
|customer_id|prediction|probability                              |
+-----------+----------+-----------------------------------------+
|1          |0.0       |[0.999242131785403,7.578682145970017E-4] |
|2          |0.0       |[0.9994171507903427,5.828492096573434E-4]|
|3          |0.0       |[0.9994708509082141,5.291490917859187E-4]|
|4          |0.0       |[0.9993295727030659,6.704272969341174E-4]|
|5          |0.0       |[0.9994678149031075,5.321850968924569E-4]|
+-----------+----------+-----------------------------------------+
only showing top 5 rows


In [0]:
import mlflow

# Log Model 2
with mlflow.start_run(run_name="Logistic_Regression_0.1"):
    mlflow.log_param("algorithm", "Logistic Regression")
    mlflow.log_param("regParam", 0.1)
    mlflow.log_param("maxIter", 50)
    mlflow.log_param("feature_count", 5)
    mlflow.log_param("training_records", 1641)
    mlflow.log_param("testing_records", 359)

    mlflow.log_metric("accuracy", 0.9972144846796658)
    mlflow.log_metric("weighted_f1", 0.9958236694988015)

# Log Model 3
with mlflow.start_run(run_name="Logistic_Regression_1.0"):
    mlflow.log_param("algorithm", "Logistic Regression")
    mlflow.log_param("regParam", 1.0)
    mlflow.log_param("maxIter", 50)
    mlflow.log_param("feature_count", 5)
    mlflow.log_param("training_records", 1641)
    mlflow.log_param("testing_records", 359)

    mlflow.log_metric("accuracy", 0.9972144846796658)
    mlflow.log_metric("weighted_f1", 0.9958236694988015)

print("Model 2 and Model 3 logged successfully!")

{"ts": "2026-11-06 20:24:11.037", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chlub3RlYm9va3MvMTM1MDEzOTcyNTY2ODA3EAEgATIkMDFhMWIyODgtMjRjZC03ZTAzLTgwOWUtZGNiMTE3ZTA4MDRmOiQyMWU5NzAxMy1kOTk3LTM4N2MtOGYzNy02ODcxMDkxM2Y4NTlKCwjV7rjXBhDA9vddUAFYAWABaKSntPjjxKMNiAEA.", "context": {}}
{"ts": "2026-11-06 20:24:11.037", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chlub3RlYm9va3MvMTM1MDEzOTcyNTY2ODA3EAEgATIkMDFhMWIyODgtMjRjZC03ZTAzLTgwOWUtZGNiMTE3ZTA4MDRmOiQyMWU5NzAxMy1kOTk3LTM4N2MtOGYzNy02ODcxMDkxM2Y4NTlKCwjV7rjXBhDA9vddUAFYAWABaKSntPjjxKMNiAEA.", "context": {}}
{"ts": "2026-11-06 20:24:11.037", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chlub3RlYm9va3MvMTM1MDEzOTcyNTY2ODA3EAEgATIkMDFhMWIyODgtMjRjZC03ZTAzLTgwOWUtZGNiMTE3ZTA4MDRmOiQyMWU5NzAxMy1kOTk3LTM4N2MtOGYzNy02ODcxMDkxM2Y4NTlK

Model 2 and Model 3 logged successfully!


In [0]:
import mlflow

# Check Model 1's experiment
experiment_1 = mlflow.get_experiment("135013972566808")

# Check Model 2 and Model 3's experiment
experiment_2 = mlflow.get_experiment("135013972566807")

print("Experiment 1:", experiment_1.name)
print("Experiment 1 ID:", experiment_1.experiment_id)

print("\nExperiment 2:", experiment_2.name)
print("Experiment 2 ID:", experiment_2.experiment_id)

Experiment 1: /Shared/Online-Shop-Churn-Prediction
Experiment 1 ID: 135013972566808

Experiment 2: /Users/khattakhooria68@gmail.com/06_Churn_Prediction
Experiment 2 ID: 135013972566807


In [0]:
all_runs = mlflow.search_runs(
    experiment_ids=[
        "135013972566807",
        "135013972566808"
    ]
)

display(
    all_runs[
        [
            "run_id",
            "tags.mlflow.runName",
            "params.regParam",
            "metrics.accuracy",
            "metrics.weighted_f1"
        ]
    ]
)

{"ts": "2026-11-06 20:41:45.185", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chlub3RlYm9va3MvMTM1MDEzOTcyNTY2ODA3EAEgATIkMDFhMWIyOTctZGUxNy03MGYwLWIxZDgtMDk4Nzg2NzljMDlhOiQyMWU5NzAxMy1kOTk3LTM4N2MtOGYzNy02ODcxMDkxM2Y4NTlKDAjb9rjXBhDAy/q/AlABWAFgAWikp7T448SjDYgBAA==.", "context": {}}
{"ts": "2026-11-06 20:41:45.185", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chlub3RlYm9va3MvMTM1MDEzOTcyNTY2ODA3EAEgATIkMDFhMWIyOTctZGUxNy03MGYwLWIxZDgtMDk4Nzg2NzljMDlhOiQyMWU5NzAxMy1kOTk3LTM4N2MtOGYzNy02ODcxMDkxM2Y4NTlKDAjb9rjXBhDAy/q/AlABWAFgAWikp7T448SjDYgBAA==.", "context": {}}
{"ts": "2026-11-06 20:41:45.185", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chlub3RlYm9va3MvMTM1MDEzOTcyNTY2ODA3EAEgATIkMDFhMWIyOTctZGUxNy03MGYwLWIxZDgtMDk4Nzg2NzljMDlhOiQyMWU5NzAxMy1kOTk3LTM4N2MtOGYzNy02ODcxMDkx

run_id,tags.mlflow.runName,params.regParam,metrics.accuracy,metrics.weighted_f1
347c0a31b1684589beb25bd6e6c26766,Logistic_Regression_1.0,1.0,0.9972144846796658,0.9958236694988015
58bb9fbd09894200b03eea8b966c2ace,Logistic_Regression_0.1,0.1,0.9972144846796658,0.9958236694988015
34366b9a526b4d909c7b8ae2773e04f3,Logistic_Regression_0.01,0.01,0.9972144846796658,0.9958236694988015
2ec9d550f54941eea035ee6db5ec1ca2,Logistic_Regression_0.01,0.01,0.9972144846796658,0.9958236694988015
af41facdf44343199d7d622d0d51e373,Logistic_Regression_0.01,0.01,0.9972144846796658,0.9958236694988015


In [0]:
experiment = mlflow.get_experiment_by_name(
    "/Shared/Online-Shop-Churn-Prediction"
)

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id]
)

display(
    runs[
        [
            "tags.mlflow.runName",
            "params.regParam",
            "metrics.accuracy",
            "metrics.weighted_f1"
        ]
    ]
)

tags.mlflow.runName,params.regParam,metrics.accuracy,metrics.weighted_f1
Logistic_Regression_0.01,0.01,0.9972144846796658,0.9958236694988015
Logistic_Regression_0.01,0.01,0.9972144846796658,0.9958236694988015
Logistic_Regression_0.01,0.01,0.9972144846796658,0.9958236694988015


In [0]:
import mlflow

# Explicitly select the shared experiment
mlflow.set_experiment("/Shared/Online-Shop-Churn-Prediction")

# Log Model 2 in the shared experiment
with mlflow.start_run(run_name="Logistic_Regression_0.1"):
    mlflow.log_param("algorithm", "Logistic Regression")
    mlflow.log_param("regParam", 0.1)
    mlflow.log_param("maxIter", 50)
    mlflow.log_param("feature_count", 5)
    mlflow.log_param("training_records", 1641)
    mlflow.log_param("testing_records", 359)

    mlflow.log_metric("accuracy", 0.9972144846796658)
    mlflow.log_metric("weighted_f1", 0.9958236694988015)

# Log Model 3 in the shared experiment
with mlflow.start_run(run_name="Logistic_Regression_1.0"):
    mlflow.log_param("algorithm", "Logistic Regression")
    mlflow.log_param("regParam", 1.0)
    mlflow.log_param("maxIter", 50)
    mlflow.log_param("feature_count", 5)
    mlflow.log_param("training_records", 1641)
    mlflow.log_param("testing_records", 359)

    mlflow.log_metric("accuracy", 0.9972144846796658)
    mlflow.log_metric("weighted_f1", 0.9958236694988015)

print("Both comparison runs saved in the shared experiment!")

Both comparison runs saved in the shared experiment!


In [0]:
shared_experiment = mlflow.get_experiment_by_name(
    "/Shared/Online-Shop-Churn-Prediction"
)

shared_runs = mlflow.search_runs(
    experiment_ids=[shared_experiment.experiment_id]
)

display(
    shared_runs[
        [
            "run_id",
            "tags.mlflow.runName",
            "params.regParam",
            "metrics.accuracy",
            "metrics.weighted_f1"
        ]
    ]
)

run_id,tags.mlflow.runName,params.regParam,metrics.accuracy,metrics.weighted_f1
a5db94460f194a4083e8c224c24074a2,Logistic_Regression_1.0,1.0,0.9972144846796658,0.9958236694988015
9ac1b16720db4f12b914aa1cabd22782,Logistic_Regression_0.1,0.1,0.9972144846796658,0.9958236694988015
34366b9a526b4d909c7b8ae2773e04f3,Logistic_Regression_0.01,0.01,0.9972144846796658,0.9958236694988015
2ec9d550f54941eea035ee6db5ec1ca2,Logistic_Regression_0.01,0.01,0.9972144846796658,0.9958236694988015
af41facdf44343199d7d622d0d51e373,Logistic_Regression_0.01,0.01,0.9972144846796658,0.9958236694988015
